## Setup
Run **Pipeline_Execution.ipynb C0 → C11 + C9c** before this notebook.  
This notebook reads from `pipeline_output/` and writes `pipeline_output/{dataset}_report.html`.  
No LLM calls are made here -- re-run any time to regenerate the report after visual changes.  
**Select kernel:** `Python (Spradley)` before running.

In [ ]:
# ── R0: Load pipeline outputs ──────────────────────────────────────────────────────
# Reads all data from pipeline_output/ -- no LLM calls in this notebook.
# Run Pipeline_Execution C0-C11 and C9c before running this cell.

import os, json
import pipeline_core

OUTPUT_DIR = pipeline_core.OUTPUT_DIR

def _load(name: str):
    path = os.path.join(OUTPUT_DIR, f"{name}.json")
    with open(path, encoding="utf-8") as f:
        return json.load(f)

clusters        = _load("clusters")
experiments     = _load("experiments")
global_store    = _load("global_store")
interview_store = _load("interview_store")
lineage         = _load("lineage")
db              = _load("db")
run_meta        = _load("run_meta")

n_iv         = run_meta["n_interviews"]
dataset_name = os.path.splitext(run_meta["input_file"])[0]

try:
    dimension_store = _load("dimension_store")
    _dim_n = dimension_store.get("_total", {}).get("interviews", 0)
    print(f"dimension_store loaded  (_total interviews: {_dim_n})")
except FileNotFoundError:
    dimension_store = {}
    print("dimension_store.json not found -- Topics radar chart will be skipped.")
    print("Run C9c in Pipeline_Execution.ipynb first to generate it.")

print(f"\nDataset:  {dataset_name}")
print(f"Interviews: {n_iv}  |  Clusters: {len(clusters)}  |  Experiments: {len(experiments)}")
print(f"\nAll outputs loaded from '{OUTPUT_DIR}'")

In [ ]:
# ── R1: Build and write {dataset}_report.html ─────────────────────────────────────────────
# Generates the full standalone HTML report and opens it in the browser.
# Tab 1 -- Report:        BCG-style findings, experiments, headline narrative.
# Tab 2 -- Topics:        Radar chart (6 emotional dimensions) + bubble map (L3 codes).
# Tab 3 -- Data Lineage:  Fully collapsible cluster -> L3 -> L2 -> L1 -> source Q&A.

import webbrowser
from pipeline_core import build_report_html

_out_path = os.path.join(OUTPUT_DIR, f"{dataset_name}_report.html")

with open(_out_path, "w", encoding="utf-8") as _f:
    _f.write(build_report_html(
        clusters, n_iv, experiments,
        global_store, interview_store, lineage, db,
        dimension_store=dimension_store,
    ))

_abs = os.path.abspath(_out_path)
print(f"Written: {_abs}")
webbrowser.open("file:///" + _abs.replace(os.sep, "/"))
print("Opened in browser.")